# Milestone 1: RAG and an agent with tools
**Project: Northwind Knowledge Assistant** | **Presenter: Nagarjuna Naidu** | Day 4, about 35 minutes

Goal: (1) answer questions from company documents with sources, (2) let an agent choose tools, (3) add one tool yourself.
Everything comes from the small `nka` package, so you will see how little code the project needs.

In [ ]:
# Optional: connect a free LLM. Remove the leading "# ", fill in your key, then restart the kernel.
# import os
# os.environ["LLM_API_KEY"] = "your-key"
# os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"   # Google AI Studio: https://generativelanguage.googleapis.com/v1beta/openai/ ; Ollama: http://localhost:11434/v1
# os.environ["LLM_MODEL"] = "llama-3.3-70b-versatile"             # any chat model your provider offers
# No key? Leave this cell as it is: everything runs in offline demo mode.

## 1. Setup: the index
48 documents are split into chunks and embedded. Old policy versions are hidden automatically.

In [ ]:
from nka.index import get_index
from nka.data import load_docs
from nka.llm_client import llm_mode, call_llm
from nka.config import MIN_SIM

idx = get_index()
print(len(load_docs()), "documents,", len(idx.chunks), "chunks,", len(idx.superseded), "old versions hidden")
print(llm_mode())

## 2. RAG: retrieve (by role), answer from the context, show sources, refuse if nothing fits

In [ ]:
def ask(question, role="employee"):
    hits = idx.search(question, k=3, role=role)
    print("Q:", question)
    if hits[0]["score"] < MIN_SIM:
        return print("A: I could not find this in the provided documents.\n")
    ctx = "\n".join(f"[S{i}] {h['text']}" for i, h in enumerate(hits, 1))
    print("A:", call_llm(f"{ctx}\n\nQUESTION: {question}", system="Answer only from the numbered context and cite [S1] etc."))
    print("  Sources:", "; ".join(dict.fromkeys(h["title"] for h in hits)), "\n")

ask("Can I work from Spain for a month?")
ask("How many weeks of parental leave do I get?")
ask("Who won the 1998 World Cup?")        # out of scope -> refusal

## 3. The agent: Thought, Action, Observation
The agent picks a tool, reads the result, and repeats until it can answer.

In [ ]:
from nka.agent import run_agent

r1 = run_agent("Which London bookings exceed the hotel cap in our travel policy, and what is the total overage?")
r2 = run_agent("What is 12.5% of 8,400?")

## 4. Guardrails
An agent can do real things, so we limit it: a **tool allow-list**, a **step limit**, and tool errors are caught (not crashed on).

In [ ]:
import inspect
from nka.tools import TOOLS

print("Allowed tools:", list(TOOLS))
print("Default step limit:", inspect.signature(run_agent).parameters["max_steps"].default)
short = run_agent("Which London bookings exceed the hotel cap in our travel policy, and what is the total overage?", max_steps=1, verbose=False)
print("With max_steps=1 the agent stops after", short["tool_calls"], "tool call.")

## 5. YOUR TASK
`lookup_policy` already exists in `nka.tools` but the agent does not know it yet. Register it, then ask: *What is POL-HR-017 about?*

In [ ]:
from nka.tools import register_tool, lookup_policy

# TODO: register_tool("lookup_policy", lookup_policy, "<one-line description>")
r = run_agent("What is POL-HR-017 about?")

**Canvas section 5:** write one line about the tools your agent needs (and which ones it must NOT have).